# 15 · Deforming domains (unfitted FEM) 🫧

We build a two-phase **rising-bubble** solver *from scratch* with **NGSolve + ngsxfem**:
* introducing every unfitted-FEM concept step by step,
* always using the *simplest* variant.
* Two **supplements** then take the production route with **ngsxditto**:
    * **(A)** a robust **mean-curvature** surface-tension solver, and
    * **(B)** a **fine-mesh** benchmark run with animation.

*This is just one of the User Meeting's many directions* — beyond `ngsxfem` lie many more faszinating extensions, insights and applications.

**Roadmap**

1. *Geometry & the level set* — background mesh, `InterpolateToP1`, `CutInfo`, `dCut`.
2. *Stationary two-phase Stokes* — doubled spaces, Nitsche coupling, surface tension, ghost penalty.
3. *Time stepping* — transporting the level set and rebuilding the cut system.
4. *The rising bubble* — putting it together.
5. *Supplementary* — the production route with `ngsxditto` (the executed fine-mesh benchmark result).

In [ ]:
from netgen.occ import WorkPlane, OCCGeometry
from ngsolve import *
from xfem import *
import numpy as np

import os
try:
    if os.environ.get("NO_WEBGUI"):       # the website ships this notebook pre-rendered
        raise ImportError                 # (matplotlib only) — webgui needs a live kernel
    from ngsolve.webgui import Draw
    from xfem import DrawDC
    HAVE_WEBGUI = True
except Exception:
    HAVE_WEBGUI = False
    def Draw(*a, **k): pass
    def DrawDC(*a, **k): pass

### The domain

A tall closed channel $[0,1]\times[0,2]$ holds a light **bubble** (the phase $\{\varphi<0\}$) inside a
heavier outer fluid ($\{\varphi>0\}$); all walls are no-slip and gravity points down. Buoyancy lifts
the bubble while surface tension keeps it compact.

![A tall closed channel with a light low-viscosity bubble (phi<0) in a heavier outer fluid (phi>0), no-slip walls, gravity pointing down](https://raw.githubusercontent.com/schruste/ngsum2026-colab/colab/data/bubble-domain.png)

### Physical parameters (Hysing–Turek, "case 1")

Index `0` is the **bubble** (the `{φ<0}` region), index `1` is the surrounding
**heavier fluid** (`{φ>0}`).  The bubble is lighter and less viscous, so
buoyancy makes it rise; surface tension keeps it compact.

In [ ]:
mu    = [1.0, 10.0]      # dynamic viscosity   [bubble, outside]
rho   = [100.0, 1000.0]  # density             [bubble, outside]
sigma = 24.5             # surface tension coefficient
g     = 0.98             # gravity
gvec  = CF((0, -g))      # gravity points downwards

R, cx, cy = 0.25, 0.5, 0.5    # initial bubble: circle of radius R centred at (cx,cy)

## 1. Geometry and the level set

In an **unfitted** (CutFEM) method the mesh does **not** resolve the interface.
We use a fixed background mesh of the whole channel `[0,1]×[0,2]` and describe
the bubble *implicitly* by the zero level set of a function

$$\varphi(x) < 0 \ \text{inside the bubble}, \qquad \varphi(x) = 0 \ \text{on the interface}, \qquad \varphi(x) > 0 \ \text{outside.}$$

We start from the signed distance of a circle.

In [ ]:
rect = WorkPlane().Rectangle(1, 2).Face()
for e in rect.edges: e.name = "wall"            # one name for all four sides
mesh = Mesh(OCCGeometry(rect, dim=2).GenerateMesh(maxh=0.06))
d = mesh.dim
Draw(mesh)

### The P1 level set: `InterpolateToP1`

`ngsxfem` allows you to integrate over the parts of each element where $\varphi < 0$, $\varphi =0$, $\varphi > 0$.  
* To make those cuts *exactly computable* it needs a level set that is **piecewise linear (P1)**
* `InterpolateToP1` takes an arbitrary expression and produces that P1 representative.

In [ ]:
levelset = sqrt((x-cx)**2 + (y-cy)**2) - R     # exact (smooth) level set
lsetp1 = GridFunction(H1(mesh, order=1))       # the P1 representative used for cutting
InterpolateToP1(levelset, lsetp1)

if HAVE_WEBGUI:
    Draw(lsetp1, mesh, "lsetp1")

### Classifying elements: `CutInfo`

`CutInfo` inspects the P1 level set and labels every element.  The useful
*domain types* are

| type      | meaning                                            |
|-----------|----------------------------------------------------|
| `NEG`     | element lies entirely in `{φ<0}` (inside)          |
| `POS`     | element lies entirely in `{φ>0}` (outside)         |
| `IF`      | element is **cut** by the interface                |
| `HASNEG`  | element has *some* `{φ<0}` part (`NEG ∪ IF`)        |
| `HASPOS`  | element has *some* `{φ>0}` part (`POS ∪ IF`)        |

`GetElementsOfType` returns a `BitArray` marking the elements of a given type.

In [ ]:
ci = CutInfo(mesh, lsetp1)
anim = GridFunction(L2(mesh, order=0), multidim=5)
for i, (t, name) in enumerate([(NEG,"NEG"),(POS,"POS"),(IF,"IF"),(HASNEG,"HASNEG"),(HASPOS,"HASPOS")]):
    print(f"{name:7s}: {ci.GetElementsOfType(t).NumSet():4d} elements")
    anim.vecs[i].FV().NumPy()[:] = ci.GetElementsOfType(t)
Draw(anim,mesh,"anim", animate=True, interpolate_multidim=False)

### Cut integration: `dCut`

`dCut(lsetp1, domain_type)` is a *differential symbol* (like `dx`) that
integrates only over the requested part of the cut elements.  Let us check it
against the known geometry of a circle of radius `R=0.25`:

$$\text{area}=\pi R^2 \approx 0.1963, \qquad \text{perimeter}=2\pi R \approx 1.5708 .$$

In [ ]:
area = Integrate(CF(1) * dCut(lsetp1, NEG), mesh)   # area of {phi<0}
peri = Integrate(CF(1) * dCut(lsetp1, IF),  mesh)   # length of the interface {phi=0}
print(f"area      = {area:.5f}   (exact {np.pi*R**2:.5f})")
print(f"perimeter = {peri:.5f}   (exact {2*np.pi*R:.5f})")

The small mismatch is exactly the **geometry error** of the P1 level set: the
true circle is approximated by straight segments.  Using a higher-order
(isoparametric) level set would reduce it — but we deliberately keep the simple
P1 variant here.

We can also *see* the unfittedness: the interface (white) cuts straight through
the background mesh, ignoring element boundaries.

In [ ]:
if HAVE_WEBGUI:
    DrawDC(lsetp1, -1.0, 1.0, mesh, "inside(-1)/outside(+1)")   # two-valued field across the cut

## 2. Stationary two-phase Stokes

Now we solve, **for a fixed interface**, the two-phase Stokes problem: in each
subdomain $i\in\{0,1\}$

$$-\operatorname{div}\,\boldsymbol\sigma_i = \rho_i\,\mathbf g, \qquad
  \operatorname{div}\mathbf u_i = 0,\qquad
  \boldsymbol\sigma_i = 2\mu_i\,\boldsymbol\varepsilon(\mathbf u_i) - p_i\,\mathbb I ,$$

coupled across the interface by

$$[\![\mathbf u]\!]=0 \quad\text{(continuous velocity)}, \qquad
  [\![\boldsymbol\sigma\,\mathbf n]\!]=\sigma\,\kappa\,\mathbf n \quad\text{(surface tension)} .$$

### Doubled / restricted spaces (Hansbo's idea)

On a **cut** element the solution has *two* branches — one for each phase.
* We take a standard Taylor–Hood pair and keep **two independent copies**,
    * one active on all `HASNEG` elements,
    * one active on all `HASPOS` elements.
* Rather than `Compress`-ing each copy down to its band, we keep the **full** doubled space and just
  switch the **active dofs** on and off (a `BitArray` intersected with the Dirichlet-free dofs). That
  is the cheap part: the space, forms and `gfup` are built **once** and reused as the interface moves
  — only `active_dofs()` is recomputed each step.

In [ ]:
order = 2                                       # Taylor-Hood P2/P1
Vb = VectorH1(mesh, order=order, dirichlet="wall")
Qb = H1(mesh, order=order-1)

# One big product space with BOTH copies living on the *full* mesh: (vel_neg, vel_pos) x
# (p_neg, p_pos) x a scalar (the 'NumberSpace' multiplier pins the pressure level). Instead of
# `Compress`-ing four restricted spaces, we keep the full space and only mark the **active dofs** —
# this lets us build the space, forms and `gfup` **once** and reuse them every step (Part 3); only
# the active-dof set changes as the interface moves.
W = FESpace([Vb*Vb, Qb*Qb, NumberSpace(mesh)], dgjumps=True)
gfup = GridFunction(W); gfup_old = GridFunction(W)   # gfup_old holds the previous velocity u^n (Part 3)
gfu, gfp, gfn = gfup.components       # gfu=(u_neg,u_pos), gfp=(p_neg,p_pos), gfn=number


def active_dofs():
    """Free dofs of W on the active band: vel/p-neg on HASNEG, vel/p-pos on HASPOS (+ the level dof)."""
    nv = GetDofsOfElements(Vb, ci.GetElementsOfType(HASNEG)); pv = GetDofsOfElements(Vb, ci.GetElementsOfType(HASPOS))
    nq = GetDofsOfElements(Qb, ci.GetElementsOfType(HASNEG)); pq = GetDofsOfElements(Qb, ci.GetElementsOfType(HASPOS))
    act = BitArray(W.ndof); act.Clear()
    oV, oQ, oN = W.Range(0).start, W.Range(1).start, W.Range(2).start
    for j in range(Vb.ndof):                       # the two velocity copies sit back-to-back in W.Range(0)
        if nv[j]: act.Set(oV + j)
        if pv[j]: act.Set(oV + Vb.ndof + j)
    for j in range(Qb.ndof):                       # the two pressure copies in W.Range(1)
        if nq[j]: act.Set(oQ + j)
        if pq[j]: act.Set(oQ + Qb.ndof + j)
    act.Set(oN)                                    # the single pressure-level dof
    return W.FreeDofs() & act                      # (Dirichlet-free) ∧ active

freedofs = active_dofs()
print("active free dofs:", sum(freedofs), "of", W.ndof)

Warning: `dgjumps=True` reserves extra couplings between neighbouring
elements for ghost-penalty stabilization (below).

### Integration regions and ghost-penalty facets

We need new integration domains 

and a set of facets/elements to be stabilized:
* `GetFacetsWithNeighborTypes` selects exactly those facets;
* `dFacetPatch` integrates over the element patch glued along each such facet.

### The weak form — built once, reused every step

We wrap the integration regions (built from the *current* `lsetp1`/`ci`, since they move with the
bubble) **and** the form into one function `build_stokes_forms()`. The fixed space `W`/`gfup` are
reused; only the cut-dependent pieces and `active_dofs()` are rebuilt each step — so the Part 3 time
loop calls the **same** `solve_stokes()` (no `Compress`, no duplicated weak form).

**Discrete variational problem.** Find $(u,p,\ell)\in W$ — the two velocity copies $u=(u_0,u_1)$, the
two pressures $p=(p_0,p_1)$ and the scalar $\ell$ — such that for all $(v,q,m)$
$$
\sum_{i=0}^{1} 2\mu_i\,(\varepsilon(u_i),\varepsilon(v_i))_{\Omega_i}
-\sum_i (p_i,\operatorname{div}v_i)_{\Omega_i}-\sum_i (q_i,\operatorname{div}u_i)_{\Omega_i}
+(\ell,q_0)_{\Omega_0}+(m,p_0)_{\Omega_0}+a_\Gamma+a_{\mathrm{gp}}
= \sum_i (\rho_i\mathbf g,v_i)_{\Omega_i}-\sigma\!\int_\Gamma (\mathbb I-\mathbf n\!\otimes\!\mathbf n){:}\nabla v_{\mathrm{avg}},
$$
with the Nitsche interface term
$a_\Gamma=\int_\Gamma\!\big(\langle\boldsymbol\sigma(u,p)\rangle\mathbf n\,[\![v]\!]+\langle\boldsymbol\sigma(v,q)\rangle\mathbf n\,[\![u]\!]+\tfrac{\lambda}{h}[\![u]\!][\![v]\!]\big)$
and the ghost penalty $a_{\mathrm{gp}}$; Part 3 adds the inertia $\sum_i\tfrac{\rho_i}{\Delta t}(u_i-u_i^{\,n},v_i)_{\Omega_i}$.

Reading it term by term (`kap` = **Hansbo cut-ratio weights** — the `NEG`-side volume fraction per
element — give the $\kappa$-weighted averages $\langle\cdot\rangle$, stable however the interface cuts):

* **per phase** — viscous $2\mu_i\,\varepsilon(u){:}\varepsilon(v)$, the velocity/pressure
  (divergence) coupling, and the buoyancy load $\rho_i\,\mathbf g\!\cdot\! v$;
* **pressure level** — the `NumberSpace` multiplier fixes $\int p$;
* **Nitsche interface** — consistent flux + adjoint + the penalty
  $\frac{\lambda}{h}[\![u]\!][\![v]\!]$ weakly enforcing $[\![u]\!]=0$;
* **surface tension** — the **Laplace–Beltrami** form
  $-\sigma\!\int_\Gamma(\mathbb I-\mathbf n\!\otimes\!\mathbf n){:}\nabla v_{\text{avg}}=\sigma\!\int_\Gamma\operatorname{div}_\Gamma v$ — the curvature force *without* ever computing $\kappa$;
* **ghost penalty** — couples each cut dof to its neighbours across the ring band (stability on
  small cuts + inf-sup).

In [ ]:
h = specialcf.mesh_size
lam   = 0.5*(mu[0]+mu[1]) * 20 * order * order        # Nitsche penalty
gp_v, gp_p = 0.1, 0.1                                  # ghost-penalty parameters


def build_stokes_forms(unsteady=False):
    "Build (a, f) for the two-phase Stokes problem on W. unsteady=True adds the (rho/dt)(u-u^n) inertia."
    dxs    = tuple(dCut(lsetp1, dom) for dom in [NEG, POS])            # bulk integrals, per phase
    dGamma = dCut(lsetp1, IF)                                          # interface integral
    dw = tuple(dFacetPatch(definedonelements=GetFacetsWithNeighborTypes(    # ghost-penalty facet patches
                   mesh, a=ci.GetElementsOfType(et), b=ci.GetElementsOfType(IF))) for et in (HASNEG, HASPOS))
    n_lset = 1.0/Norm(grad(lsetp1)) * grad(lsetp1)                     # interface normal
    kap = [CutRatioGF(ci), 1.0 - CutRatioGF(ci)]                       # Hansbo cut-ratio weights
    P = Id(d) - OuterProduct(n_lset, n_lset)                           # tangential projection
    u, p, num = W.TrialFunction(); v, q, m = W.TestFunction()
    uold = gfup_old.components[0]                                      # previous velocity u^n = (u_neg, u_pos)
    def eps(w):          return 0.5*(Grad(w) + Grad(w).trans)
    def sig(i, w, pp):   return -2*mu[i]*eps(w[i]) + pp[i]*Id(d)
    def avg_flux(w, pp): return sum(kap[i]*sig(i, w, pp)*n_lset for i in range(2))
    def avg_grad(w):     return sum(kap[1-i]*Grad(w[i]) for i in range(2))
    def jump(w):         return w[0] - w[1]
    a = BilinearForm(W, symmetric=False); f = LinearForm(W)
    for i in [0, 1]:                                                   # per phase: viscous + div-coupling + buoyancy
        a += 2*mu[i]*InnerProduct(eps(u[i]), eps(v[i])) * dxs[i]
        a += (-div(u[i])*q[i] - div(v[i])*p[i]) * dxs[i]
        f += rho[i]*gvec*v[i] * dxs[i]
        if unsteady:                                                  # backward-Euler inertia (rho/dt)(u - u^n)
            a += rho[i]/dt * InnerProduct(u[i], v[i]) * dxs[i]
            f += rho[i]/dt * InnerProduct(uold.components[i], v[i]) * dxs[i]
    a += (num*q[0] + m*p[0]) * dxs[0]                                  # pin the pressure level
    a += (avg_flux(u, p)*jump(v) + avg_flux(v, p)*jump(u)) * dGamma    # Nitsche consistency + adjoint
    a += lam/h * jump(u)*jump(v) * dGamma                              # Nitsche penalty
    f += -sigma * InnerProduct(P, avg_grad(v)) * dGamma                # Laplace-Beltrami surface tension
    for i in [0, 1]:                                                   # ghost penalty (velocity & pressure)
        a += gp_v/h**2 * (u[i]-u[i].Other())*(v[i]-v[i].Other()) * dw[i]
        a += -gp_p     * (p[i]-p[i].Other())*(q[i]-q[i].Other()) * dw[i]
    return a, f


def solve_stokes(unsteady=False):
    "Build, assemble and solve the two-phase Stokes system on the current geometry (updates gfup)."
    if unsteady:
        gfup_old.vec.data = gfup.vec                                  # remember u^n before overwriting
    a, f = build_stokes_forms(unsteady)
    with TaskManager():
        a.Assemble(); f.Assemble()
        gfup.vec.data = a.mat.Inverse(active_dofs(), inverse="umfpack") * f.vec   # invert on the active band
    return gfup

In [ ]:
solve_stokes()

### Does it make sense?  The Laplace law

A clean check that the surface tension is implemented correctly is the **Laplace
law**: across a circular interface the pressure jump must be

$$p_{\text{in}} - p_{\text{out}} = \sigma\,\kappa = \frac{\sigma}{R} = \frac{24.5}{0.25} = 98 .$$

(We measure it *locally on the interface*; comparing volume-averaged pressures
would be polluted by the hydrostatic gradient from gravity.)

In [ ]:
dGamma = dCut(lsetp1, IF)                                   # the interface measure, for this check
per   = Integrate(CF(1)*dGamma, mesh)
pjump = Integrate((gfp.components[0]-gfp.components[1])*dGamma, mesh) / per
print(f"interface-averaged (p_in - p_out) = {pjump:6.2f}   (Laplace sigma/R = {sigma/R:.1f})")

def draw_solution(title):
    "Glue the two phases into single-valued scalars and show u_x, u_y, |u|, p as one multidim scene."
    uc = IfPos(lsetp1, gfu.components[1], gfu.components[0])   # velocity — pick the active phase
    pc = IfPos(lsetp1, gfp.components[1], gfp.components[0])   # pressure — pick the active phase
    Vs = H1(mesh, order=1)                                     # order 1 keeps the webgui scene small
    show = GridFunction(Vs, multidim=0)
    for cf in [uc[0], uc[1], Norm(uc), pc]:                    # x-velocity, y-velocity, |u|, pressure
        g = GridFunction(Vs); g.Set(cf)
        show.AddMultiDimComponent(g.vec)
    Draw(show, mesh, title)

if HAVE_WEBGUI:
    # DrawDC draws a *two-valued* field (ragged across the cut); instead glue each field into a
    # single-valued scalar and show four of them in one scene — drag the multidim slider: u_x·u_y·|u|·p
    draw_solution("stationary solution — u_x · u_y · |u| · p")

## 3. Time stepping: moving the interface

So far the interface was frozen.  To let the bubble rise, the level set must be
**transported** by the computed velocity field $\mathbf u$:

$$\partial_t \varphi + \mathbf u\cdot\nabla\varphi = 0 .$$

We discretise this with the simplest robust explicit scheme — **upwind
Discontinuous Galerkin** on an `L2` field, advanced by explicit Euler.  Because
the velocity is (almost) divergence free we may use the conservative flux form.

In [ ]:
fes_phi = L2(mesh, order=2, dgjumps=True)
phi = GridFunction(fes_phi)
phi.Set(levelset)                            # initial level set in the transport space

gf_w = GridFunction(VectorH1(mesh, order=order))   # holds the advecting velocity each step

u_l, v_l = fes_phi.TnT()
n_F  = specialcf.normal(d)
flux = gf_w * n_F
upw  = IfPos(flux, u_l, u_l.Other())         # upwind value across a facet
conv = BilinearForm(fes_phi, nonassemble=True)       # 'nonassemble' -> we only use .Apply()
conv += -u_l * (gf_w * grad(v_l)) * dx
conv += flux * upw * (v_l - v_l.Other()) * dx(skeleton=True)   # interior facets
conv += IfPos(flux, flux*u_l, 0) * v_l * ds(skeleton=True)     # outflow boundary
invm = fes_phi.Mass(1).Inverse()             # L2 mass is block diagonal -> cheap inverse
res  = phi.vec.CreateVector()

def transport(dt_total, nsub):
    "Advance phi by dt_total using nsub explicit sub-steps, then refresh the P1 cut field."
    for _ in range(nsub):
        conv.Apply(phi.vec, res)
        phi.vec.data -= (dt_total/nsub) * invm * res
    lsetp1.Set(phi)        # project the (discontinuous) transported field to the P1 cut field
    ci.Update(lsetp1)      # re-classify elements for the new interface position

We sub-cycle the transport (`nsub` small steps per Stokes solve) to respect the
explicit CFL limit cheaply — the Stokes solve is the expensive part, the
transport is not.

### One reusable Stokes solve

Because the space `W`, the field `gfup`, the form-builder `build_stokes_forms()` and `active_dofs()`
were all set up in Part 2, there is **nothing heavy to rebuild** here: each step we just call the
same `solve_stokes()`, which re-derives the cut regions and the active dofs from the *current*
`lsetp1`/`ci` and re-assembles the light forms. The doubled space is never re-created — only the
active-dof set changes as the bubble moves.

**And it really does change every step.** The doubled dofs live on the **cut band**; as the interface
moves, *different* elements get cut, so `active_dofs()` returns a different set. To see it, prescribe
two nearby interface positions (a fictitious step) and mark the cut elements — the band slides up with
the bubble:

In [ ]:
# a sketch...
import matplotlib.pyplot as plt
verts = np.array([list(v.point) for v in mesh.vertices])
tris  = np.array([[vv.nr for vv in el.vertices] for el in mesh.Elements(VOL)])
lset_demo = GridFunction(H1(mesh, order=1))
fig, ax = plt.subplots(1, 2, figsize=(6.4, 5.2))
for k, cyk in enumerate([0.50, 0.56]):                       # two close level sets = one fictitious step
    InterpolateToP1(sqrt((x-cx)**2 + (y-cyk)**2) - R, lset_demo); ci_demo = CutInfo(mesh, lset_demo)
    cut = ci_demo.GetElementsOfType(IF)                      # the cut elements carry the doubled dofs
    col = np.array([1.0 if cut[el.nr] else 0.0 for el in mesh.Elements(VOL)])
    ax[k].tripcolor(verts[:,0], verts[:,1], tris, facecolors=col, cmap="Oranges", vmin=0, vmax=1.4)
    ax[k].triplot(verts[:,0], verts[:,1], tris, color="k", lw=0.15, alpha=0.25)
    th = np.linspace(0, 2*np.pi, 120); ax[k].plot(cx + R*np.cos(th), cyk + R*np.sin(th), "b-", lw=1.6)
    ax[k].set_xlim(0.12, 0.88); ax[k].set_ylim(0.18, 0.92); ax[k].set_aspect("equal")
    ax[k].set_title(f"interface at $y_c$={cyk}\ncut elements (orange)", fontsize=9)
    ax[k].set_xticks([]); ax[k].set_yticks([])
fig.suptitle("Active dofs move with the interface — the cut band shifts each step", fontsize=10)
fig.tight_layout()

### The time loop

The algorithm of one step is:

1. **solve** the *unsteady* two-phase Stokes problem on the current geometry → velocity $\mathbf u$
   (`solve_stokes(unsteady=True)` adds the backward-Euler inertia $\tfrac{\rho_i}{\Delta t}(\mathbf u-\mathbf u^n)$, with $\mathbf u^n$ the previous step's velocity in `gfup_old`);
2. build the single advecting field $\mathbf u = \mathbf u_{\text{pos}}$ outside, $\mathbf u_{\text{neg}}$ inside;
3. **transport** the level set and **re-classify** the mesh;
4. repeat.

We monitor the bubble's centroid height, its rise velocity, and its area (mass
conservation) — a good unfitted scheme keeps the area nearly constant *without*
any reinitialisation over this time span.

In [ ]:
dt, tend, nsub = 0.005, 1.0, 5      # the simple level set stays clean to ~t=1; see note below
nsteps = int(tend/dt + 0.5)

A0 = Integrate(CF(1)*dCut(lsetp1, NEG), mesh)
hist = {"t": [0.0], "yc": [Integrate(y*dCut(lsetp1,NEG),mesh)/A0], "area": [A0]}
shapes = [(0.0, lsetp1.vec.FV().NumPy().copy())]
print(f"t=0.000  y_c={hist['yc'][0]:.4f}  area={A0:.5f}")

with TaskManager():
    for step in range(1, nsteps+1):
        gfup = solve_stokes(unsteady=True)
        gu = gfup.components[0]
        gf_w.Set(IfPos(lsetp1, gu.components[1], gu.components[0]))
        transport(dt, nsub)

        t = step*dt
        A  = Integrate(CF(1)*dCut(lsetp1, NEG), mesh)
        yc = Integrate(y*dCut(lsetp1, NEG), mesh)/A
        hist["t"].append(t); hist["yc"].append(yc); hist["area"].append(A)
        if abs(t - round(t*4)/4) < dt/2:        # snapshot every 0.25 time units
            shapes.append((t, lsetp1.vec.FV().NumPy().copy()))
        if step % 20 == 0 or step == nsteps:
            vy = Integrate(gu.components[0][1]*dCut(lsetp1,NEG),mesh)/A
            print(f"t={t:5.3f}  y_c={yc:6.4f}  rise v_y={vy:6.3f}  "
                  f"area={A:.5f} ({100*(A-A0)/A0:+.2f}%)")
print("done")

## 4. The rising bubble

Finally we look at the result. The webgui scene below animates the **interface** (the level-set
zero) rising and flattening into a cap; the two plots show the centroid height climbing roughly
linearly and the bubble area staying nearly constant.

In [ ]:
# the interface over time as a webgui level-set animation (rebuilt from the saved snapshots):
# its zero contour is the bubble surface rising and flattening into a cap.
Vl1 = H1(mesh, order=1)
lset_anim = GridFunction(Vl1, multidim=0)
for t_, lv in shapes:
    g = GridFunction(Vl1); g.vec.FV().NumPy()[:] = lv
    lset_anim.AddMultiDimComponent(g.vec)
if HAVE_WEBGUI:
    Draw(lset_anim, mesh, "interface φ over time (drag the slider; the zero level set is the bubble)",
         interpolate_multidim=True, animate=True)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].plot(hist["t"], hist["yc"]); ax[0].set_xlabel("t"); ax[0].set_ylabel("centroid height")
ax[0].set_title("bubble rises"); ax[0].grid(alpha=.3)
ax[1].plot(hist["t"], 100*(np.array(hist["area"])-A0)/A0)
ax[1].set_xlabel("t"); ax[1].set_ylabel("area change [%]")
ax[1].set_title("mass conservation"); ax[1].grid(alpha=.3)
fig.tight_layout()

## Summary

We built a complete two-phase rising-bubble solver with nothing but NGSolve and
ngsxfem, introducing each unfitted-FEM ingredient on the way:

* **`InterpolateToP1` + `CutInfo`** — implicit geometry and element marking;
* **`dCut` / `dFacetPatch`** — integration over cut regions and facet patches;
* **Partially deactivated doubled spaces** — the CutFEM representation of a two-phase field;
* **Nitsche coupling** with **Hansbo (`CutRatioGF`) weights** — the interface conditions;
* **Laplace–Beltrami** surface tension characterization;
* **ghost penalty** on the **ring band** — stability on small cuts, inf-sup and solution extension;
* **explicit upwind-DG transport** + rebuild — the moving-interface time loop.

Every choice was the *simplest* one, and the limitations are exactly the ones
you would expect from that:

* the **P1 geometry** leaves a visible area error (~1 %) and produces *parasitic
  currents* at the interface (the surface-tension force and the discrete pressure
  gradient do not balance exactly);
* without **reinitialisation** the transported level set slowly drifts away from
  a signed-distance function, so beyond `t ≈ 1` the volume is no longer conserved
  and the interface starts to roughen — which is why we stop at `t = 1`.

Natural next steps for accuracy each slot directly into the framework above: 
* a higher-order **isoparametric** level set (smaller geometry error, far weaker
parasitic currents),
* a **space–time** discretisation of the transport,
* and periodic **reinitialisation** of the level set for long-time runs.

The supplement below makes this concrete: it solves the *same* problem with the
high-level **ngsxditto** library — which bundles exactly those improvements
(isoparametric geometry, reinitialisation, a proper curvature solver) — and uses
it to run the full **Hysing–Turek benchmark** with a smooth animation.

## Supplementary â the production route (`ngsxditto`)

*(Supplementary â needs the **ngsxditto** add-on.)* Everything above we built *by hand* to expose the
concepts. In practice one reaches for a library: **`ngsxditto`** bundles exactly the production-grade
choices we skipped, at the **same** Taylor–Hood P2/P1 order:

| our hand-built notebook            | `ngsxditto`                                      |
|------------------------------------|--------------------------------------------------|
| P1 level set (geometry error)      | **isoparametric** level set (`LevelSetGeometry`) |
| explicit DG transport, no reinit   | DG transport **+ FastMarching reinitialisation** |
| Laplace–Beltrami (parasitic curr.) | dedicated **`MeanCurvatureSolver`**         |
| rebuild the cut system every step  | incremental `TwoPhaseTaylorHood` + `TimeLoop`    |

It is pinned to a development build the public site cannot reproduce, so we do not run it here.
Instead, the **executed fine-mesh result** is below: a `ngsxditto` run to `t ≈ 2`, the interface
and interior velocity as the bubble rises and flattens into the clean ellipsoidal cap — no
roughening, no volume loss, matching the **Hysing–Turek** reference (max rise velocity
≈ 0.242 at `t ≈ 0.92`, min circularity ≈ 0.901 at `t ≈ 1.9`, centroid `y(3) ≈ 1.081`).

![Fine-mesh ngsxditto rising bubble — interface and velocity field from t=0 to t≈2](https://raw.githubusercontent.com/schruste/ngsum2026-colab/colab/data/rising_bubble_ngsxditto.gif)

## Conclusion

Geometry, coefficient functions, spaces, weak forms, solvers, time stepping,
nonlinearity, coupling, PDEs on curved surfaces, and now a moving phase front ...

And now the real **Grand Expedition** starts:
the rest of the **NGSolve User Meeting** and its
fantastic contributions!

![The pirate leads a caravan of riders on rainbow mesh tori toward the mountains —
the Grand Expedition.](data/expedition.jpg)

In [ ]:
# Navigation between units — shown only in a live notebook (Colab / JupyterLite /
# local Jupyter), never in the rendered website (which has its own prev/next nav).
import os, sys
if not os.environ.get("WEBGUI_SCENE_DIR"):          # not the static site build
    _prev = ("14-melting-chocolate", "14 · Melting the chocolate 🍫☕")
    _next = None
    def _u(_nb):
        if "google.colab" in sys.modules:
            return "https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/" + _nb + ".ipynb"
        return _nb + ".ipynb"                       # JupyterLite & local: relative .ipynb link
    _parts  = ["⬅️ **Previous:** [%s](%s)" % (_prev[1], _u(_prev[0]))] if _prev else []
    _parts += ["➡️ **Next:** [%s](%s)" % (_next[1], _u(_next[0]))] if _next else []
    from IPython.display import display, Markdown
    display(Markdown(" · ".join(_parts)))